# Use Case — Prioritizing Cooling Resources for Vulnerable Populations

**Who this is for**  
Public-health directors, school-district superintendents, social-services leads, and climate-resilience officers responsible for protecting vulnerable populations (children, elderly, chronic-illness patients) during heat waves.

**The scenario**  
A heat-wave advisory is active for the next 72 hours. You have a limited pool of cooling-center funding, wellness-check staff, and emergency-response capacity. You cannot cover every facility equally — you need to direct resources to facilities that combine **the hottest locations** with **the largest vulnerable populations**.

This notebook combines **your facility registry** (points + vulnerability counts) with **FortyGuard layers** to answer three questions:

1. **Where is it hottest?**  ← heatmap × your facility points
2. **Where is the exposure worst** when we weight by population at risk?  ← your vulnerability column × tile temperature
3. **What does the day look like at the worst facility?**  ← environmental parameters profile

Output: a prioritized action list, one row per facility, with a type-specific recommendation (cooling-center activation, HVAC retrofit, wellness checks, heat-advisory alert list).

> **Bring your own data.** Sample data at `data/sample_public_facilities.csv`. Swap it out — as long as the columns match (`facility_id`, `name`, `type`, `vulnerable_population`, `latitude`, `longitude`), everything downstream works.

**What makes this pattern different from a simple hotspot map**: the user data carries a *weight* (`vulnerable_population`) that modifies the ranking. The hottest block is not always the highest-priority block — a 32 °C facility serving 540 children outranks a 34 °C facility serving 140.

---

## Setup

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import pandas as pd
import folium
import matplotlib.pyplot as plt
from shapely.geometry import Point, shape

from fortyguard import FortyGuardClient
from fortyguard.samples import SAN_JOSE_POLYGON

client = FortyGuardClient()

AOI           = SAN_JOSE_POLYGON         # ~104 km² (~40 mi²) across central San Jose
STUDY_DATE    = '2024-07-15'
STUDY_HOUR    = '14:00'
GRANULARITY_M = 100                      # 100 m keeps the heatmap tile count tractable over this AOI
BASELINE_C    = 28.0                     # temperatures above this count toward risk

print(f'Authenticated to {client.base_url}')

---
## Step 1 — Load the facility registry

### What you are doing
Reading the list of facilities and the vulnerable-population count each one serves. The `type` column lets us tailor recommendations downstream — a school gets a different intervention from a clinic.

### Why this matters
The vulnerability weight is *your* field, not ours. Two departments can run this same notebook with different data — the emergency-services team counts chronic-illness patients, the school district counts enrolled pupils — and get answers that reflect their own mandate.

In [ ]:
facilities = pd.read_csv(ROOT / 'data' / 'sample_public_facilities.csv')
print(f"Loaded {len(facilities)} facilities serving "
      f"{facilities['vulnerable_population'].sum():,} vulnerable people")
facilities.groupby('type')[['vulnerable_population']].agg(['count', 'sum'])

---
## Step 2 — Generate the heat layer and attach temperature to each facility

### What you are doing
One heatmap call over the coverage AOI. Then a point-in-tile spatial join so every facility row gets a `temperature_c` column.

### Why this matters
Before this step, heat is a general condition. After this step it is an attribute on each row of your operational table — ready to be sorted, filtered, or joined to anything else you manage.

In [ ]:
heatmap = client.create_heatmap(
    polygon_aoi=AOI, start_date=STUDY_DATE, start_time=STUDY_HOUR,
    filter_type=1, granularity=GRANULARITY_M,
)
map_data = heatmap['result'].get('map_data') or {}
features = map_data.get('features', []) if isinstance(map_data, dict) else []
tile_polys = [(shape(f['geometry']), f['properties'].get('temperature')) for f in features]

def _temp_at(lat, lon):
    p = Point(lon, lat)
    for poly, t in tile_polys:
        if poly.contains(p): return t
    return min(tile_polys, key=lambda pt: pt[0].centroid.distance(p))[1] if tile_polys else None

facilities['temperature_c'] = facilities.apply(
    lambda r: _temp_at(r.latitude, r.longitude), axis=1)
facilities[['facility_id', 'name', 'type',
            'vulnerable_population', 'temperature_c']].head()

---
## Step 3 — Compute exposure-weighted risk

### What you are doing
Combining the temperature with the vulnerable-population count into a single risk figure per facility:

```
exposure_risk = max(temperature_c − baseline, 0) × vulnerable_population
```

Baseline = 28 °C (adjust per your local climate). The subtraction zeros out facilities in comfortable tiles so population alone cannot rank them up.

### Why this matters
This is the whole point of bringing your own data. A purely thermal ranking would put a sparsely-used block above an elementary school when the school's block is only one degree cooler. Exposure-weighting resolves that — the ranking reflects *people at risk*, not only thermometer readings.

In [ ]:
facilities['heat_excess_c'] = (facilities['temperature_c'] - BASELINE_C).clip(lower=0).round(2)
facilities['exposure_risk'] = (facilities['heat_excess_c']
                               * facilities['vulnerable_population']).round(0)
facilities = facilities.sort_values('exposure_risk', ascending=False).reset_index(drop=True)
facilities.insert(0, 'rank', facilities.index + 1)
facilities[['rank', 'facility_id', 'name', 'type',
            'vulnerable_population', 'temperature_c',
            'heat_excess_c', 'exposure_risk']]

---
## Step 4 — Profile peak heat at the highest-risk facility

### What you are doing
Calling environmental parameters at the top-ranked facility to get heat index and humidity across the day. We plot the diurnal profile and extract the peak hour.

### Why this matters
"Activate a cooling center" is not a plan until you know *when* to open it. If peak discomfort is 13:00–16:00, a 10:00 opening wastes staff hours. If it is 14:00–19:00, a 16:00 closing leaves people stranded during the afternoon rush.

In [ ]:
top = facilities.iloc[0]
env = client.environmental_parameters(
    latitude=top.latitude, longitude=top.longitude,
    temperature=float(top.temperature_c),
    start_date=STUDY_DATE, start_time='07:00', end_time='20:00',
    filter_type=2, verbose=False,
)
res  = env['result']
loc  = res['locations'][0]
pars = loc.get('parameters', {})
ts   = pd.to_datetime(res['metadata'].get('timestamps', []))

env_df = pd.DataFrame({k: v for k, v in pars.items()
                       if isinstance(v, list) and len(v) == len(ts)})
env_df.insert(0, 'timestamp', ts); env_df.set_index('timestamp', inplace=True)

thermal = [c for c in ('heat_index_celsius', 'apparent_temperature_celsius',
                       'wet_bulb_temperature_celsius') if c in env_df.columns]
if thermal:
    env_df[thermal].plot(figsize=(9, 3), marker='o')
    plt.axhline(32, color='red', linestyle='--', alpha=0.5, label='discomfort threshold')
    plt.title(f"Heat profile at #{top['rank']}  {top['name']} ({top['type']})")
    plt.ylabel('°C'); plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

peak_hi = env_df['heat_index_celsius'].max() if 'heat_index_celsius' in env_df.columns else None
peak_t  = env_df['heat_index_celsius'].idxmax() if 'heat_index_celsius' in env_df.columns else None
print(f"Peak heat index: {peak_hi} °C at {peak_t}")

---
## Step 5 — Priority map

### What you are doing
Rendering the ranking as a map you can hand to an incident commander. Facility markers are colored by risk quartile and **sized by vulnerable population** so the map itself communicates both signals at once.

### Why this matters
A table is for analysts. A map is for the operations room.

In [ ]:
def _tier(r):
    if r >= facilities['exposure_risk'].quantile(0.75): return 'critical', '#b30000'
    if r >= facilities['exposure_risk'].quantile(0.50): return 'high',     '#e34a33'
    if r >= facilities['exposure_risk'].quantile(0.25): return 'moderate', '#fdae61'
    return 'low', '#1a9850'

facilities['priority'] = facilities['exposure_risk'].apply(lambda r: _tier(r)[0])

lo, hi = (min(t for _, t in tile_polys) if tile_polys else 0,
          max(t for _, t in tile_polys) if tile_polys else 1)
def _tile_style(feat):
    t = feat['properties'].get('temperature', lo)
    frac = 0 if hi == lo else (t - lo) / (hi - lo)
    r, b = int(255*frac), int(255*(1-frac))
    return {'fillColor': f'#{r:02x}00{b:02x}', 'color': '#00000000', 'fillOpacity': 0.30, 'weight': 0}

center = [facilities['latitude'].mean(), facilities['longitude'].mean()]
fmap = folium.Map(location=center, zoom_start=15, tiles='cartodbpositron')
if features:
    folium.GeoJson(map_data, style_function=_tile_style).add_to(fmap)
for _, r in facilities.iterrows():
    _, color = _tier(r['exposure_risk'])
    radius = max(5, (r['vulnerable_population'] ** 0.5) / 2)
    folium.CircleMarker(
        location=[r.latitude, r.longitude],
        radius=radius, color='black', weight=1,
        fill=True, fill_color=color, fill_opacity=0.9,
        popup=(f"#{r['rank']} {r['facility_id']} — {r['name']}<br/>"
               f"type: {r['type']}<br/>"
               f"population: {r['vulnerable_population']}<br/>"
               f"temp: {r.temperature_c:.1f} °C<br/>"
               f"risk: {r['exposure_risk']:.0f}"),
    ).add_to(fmap)
fmap

---
## Step 6 — Type-aware action list

### What you are doing
Translating the risk tier into a concrete recommended action **that depends on facility type**. Rules:

| Type | Critical / High | Moderate | Low |
|------|-----------------|----------|-----|
| School / Daycare / After-School | Cooling-center activation + HVAC retrofit CapEx | AC-check before summer | Monitor |
| Clinic / Health Center | Extended hours, tele-health surge, heat-illness triage | Staff-brief protocol | Monitor |
| Senior Center / Eldercare | Wellness check-ins, door-to-door | Welfare-call list | Monitor |
| Community Center / Library | Designate as public cooling shelter | Publicize opening hours | Monitor |

### Why this matters
Generic "high risk" is not an action — an operations team needs a verb matched to the resource that department actually controls. The recommendation column is pre-filtered per facility type so every row is immediately assignable.

In [ ]:
ACTION_MATRIX = {
    ('School',           'critical'): 'Cooling-center activation + HVAC retrofit CapEx',
    ('School',           'high'):     'Cooling-center activation + HVAC retrofit CapEx',
    ('Daycare',          'critical'): 'Cooling-center activation + HVAC retrofit CapEx',
    ('Daycare',          'high'):     'Cooling-center activation + HVAC retrofit CapEx',
    ('After-School',     'critical'): 'Cooling-center activation + HVAC retrofit CapEx',
    ('After-School',     'high'):     'Cooling-center activation + HVAC retrofit CapEx',
    ('Clinic',           'critical'): 'Extended hours + tele-health surge + heat-illness triage',
    ('Clinic',           'high'):     'Extended hours + tele-health surge + heat-illness triage',
    ('Senior Center',    'critical'): 'Wellness check-ins, door-to-door',
    ('Senior Center',    'high'):     'Wellness check-ins, door-to-door',
    ('Eldercare',        'critical'): 'Wellness check-ins, door-to-door',
    ('Eldercare',        'high'):     'Wellness check-ins, door-to-door',
    ('Community Center', 'critical'): 'Designate as public cooling shelter',
    ('Community Center', 'high'):     'Designate as public cooling shelter',
    ('Library',          'critical'): 'Designate as public cooling shelter',
    ('Library',          'high'):     'Designate as public cooling shelter',
}
MODERATE_BY_TYPE = {
    'School': 'Pre-summer AC audit', 'Daycare': 'Pre-summer AC audit', 'After-School': 'Pre-summer AC audit',
    'Clinic': 'Staff heat-illness brief',
    'Senior Center': 'Welfare-call list prep', 'Eldercare': 'Welfare-call list prep',
    'Community Center': 'Publicize shelter hours', 'Library': 'Publicize shelter hours',
}

def _action(row):
    key = (row['type'], row['priority'])
    if key in ACTION_MATRIX: return ACTION_MATRIX[key]
    if row['priority'] == 'moderate':
        return MODERATE_BY_TYPE.get(row['type'], 'Monitor')
    return 'Monitor'

facilities['recommended_action'] = facilities.apply(_action, axis=1)

cols = ['rank', 'facility_id', 'name', 'type', 'vulnerable_population',
        'temperature_c', 'exposure_risk', 'priority', 'recommended_action']
facilities[cols]

In [ ]:
out = ROOT / 'outputs' / 'vulnerable_facility_action_list.csv'
out.parent.mkdir(parents=True, exist_ok=True)
facilities[cols].to_csv(out, index=False)
print(f'Saved to {out}')
print(f"Critical/High facilities: "
      f"{(facilities['priority'].isin(['critical','high'])).sum()} of {len(facilities)} "
      f"covering {facilities.loc[facilities['priority'].isin(['critical','high']),'vulnerable_population'].sum():,} people")

---
## Wrap-up

Starting from a facility registry CSV with a vulnerability column you now have:

| Artifact | Audience |
|----------|----------|
| Temperature-joined facility table | Public-health analytics |
| Exposure-weighted priority ranking | Emergency operations |
| Diurnal heat profile at top-risk facility | Staffing / opening-hours planning |
| Risk-tier priority map | Incident command briefing |
| Type-aware action list CSV | Department heads |

**Apply this pattern to adjacent use cases**: the `vulnerable_population` column can become any weighted exposure — student-hours outdoors (school PE scheduling), patient-visit counts (clinic staffing), chronic-illness density (public-health early warning). The workflow — weighted points × heatmap → ranked action list — carries over unchanged.